# AAT on Kaggle: Adaptive Adversarial Training rebuild

**Settings:** Accelerator = *GPU T4 x2*, Internet = *On*, Persistence = *Files only*.

Each session: run the setup cell, then **one** stage cell. Checkpoints resume automatically from `runs/<name>_s<seed>/last.pt`, so a session that times out can be re-run.
After each stage, save the notebook version ("Save & Run All" or "Quick Save"), or attach `runs/` as a Dataset so later sessions can find the pretrained model.

In [ ]:
%cd /kaggle/working
!git clone -q -b matt/funny-planck-3j7md5 https://github.com/mattobryan/AAT.git 2>/dev/null || (cd AAT && git pull -q)
%cd /kaggle/working/AAT
!pip install -q git+https://github.com/fra31/auto-attack
import torch; print(torch.__version__, torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Optional: reuse runs/ from an earlier session that was saved as a Kaggle Dataset
import os, shutil
SRC = '/kaggle/input/aat-runs/runs'   # <- change to your dataset path
if os.path.exists(SRC) and not os.path.exists('runs'):
    shutil.copytree(SRC, 'runs'); print('restored runs/')

## Day 1: smoke test and timing (≈20 min). Fill in the budget table in the README from `epoch_time_s`.

In [ ]:
!python -m pytest -q tests/test_core.py
!python -m aat.train --config configs/v3a_max.yaml --set name=timing_max init_from=null train.epochs=1 limit_batches=50 monitor.every=0
!python -m aat.train --config configs/v2a_cyclic.yaml --set name=timing_single init_from=null train.epochs=1 limit_batches=50 monitor.every=0
!tail -n1 runs/timing_max_s0/log.jsonl runs/timing_single_s0/log.jsonl | grep -o '"epoch_time_s": [0-9.]*'

## Stage 0: shared ℓ∞-AT pretrain (~1–1.5 h on one T4)

In [ ]:
!python -m aat.train --config configs/pretrain_linf.yaml

## Stage 1: baselines and ablations, seed 0 (two runs in parallel, one per GPU)

In [ ]:
!bash scripts/run_queue.sh "ramp eat v3a_max v1a_linear" "seed=0"
!bash scripts/run_queue.sh "v1b_sample v3b_classeps v2a_cyclic v2b_adaptive" "seed=0"
!bash scripts/run_queue.sh "aat_full aat_full_no_l2 v3b_thesis" "seed=0"

## Stage 2: extra seeds for the headline comparison

In [ ]:
for s in (1, 2):
    !bash scripts/run_queue.sh "ramp eat v3a_max aat_full v3b_classeps v2b_adaptive" "seed={s}"

## Stage 3: AutoAttack evaluation at fixed ε, plus the unseen-threat grid

In [ ]:
import glob, subprocess, os
runs = sorted(r for r in glob.glob('runs/*_s*') if os.path.exists(f'{r}/final.pt') and 'timing' not in r and 'pretrain' not in r)
todo = [r for r in runs if not os.path.exists(f'{r}/eval_autoattack.json')]
print(len(todo), 'to evaluate')
half = [todo[0::2], todo[1::2]]
procs = [subprocess.Popen(['python', '-m', 'aat.evaluate', '--run', *h], env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(g)},
                          stdout=open(f'eval_gpu{g}.log', 'w'), stderr=subprocess.STDOUT) for g, h in enumerate(half) if h]
[p.wait() for p in procs]

In [ ]:
!python scripts/aggregate.py --runs runs --tag autoattack
from IPython.display import Image, Markdown, display
display(Markdown(open('results/summary_autoattack.md').read())); display(Image('results/tradeoff_autoattack.png'))